# Notebook 1 — Data Pipeline, Metadata, Splits and Configuration Handoff

## Purpose of this notebook

This notebook prepares the data pipeline for the weakly supervised Video Anomaly Detection project. It does not train the final model. Instead, it sets up dataset paths, metadata tables, train/validation/test splits, fixed-length temporal feature bags, and configuration files used by the later notebooks.

The project uses two datasets:

1. **UCF-Crime**  
   UCF-Crime is used as the main surveillance anomaly dataset. In this project, UCF-Crime is stored as extracted PNG frames. The notebook scans these frames, groups them into video clips, assigns binary normal/anomaly labels, and creates split files.

2. **XD-Violence**  
   XD-Violence is used through pre-extracted I3D RGB and optical-flow feature files. The notebook pairs RGB and Flow `.npy` files, infers binary weak labels, builds metadata, and creates fixed-length feature bags for later MIL training.

The overall purpose of this notebook is to transform the raw dataset structure into a clean, reproducible format that can be consumed by the feature loading, training and evaluation notebooks.

## Pipeline overview

```text
Raw UCF PNG frames / XD I3D features
        ↓
Clip-level metadata
        ↓
Binary normal/anomaly labels
        ↓
Train / validation / test splits
        ↓
Fixed-length temporal feature bags
        ↓
Saved configuration for later notebooks
```

The project follows a **weakly supervised** formulation. Exact frame-level anomaly boundaries are not used for training. Instead, each video or clip is labelled as normal or anomalous. Later notebooks use Multiple Instance Learning, where a video is represented as a bag of temporal segment features.

## Dataset format

### UCF-Crime frame format

UCF-Crime is stored as extracted PNG frames rather than raw video files.

Example paths:

```text
datasets/ucf-crime-dataset/Train/Arson/Arson042_x264_1170.png
datasets/ucf-crime-dataset/Train/NormalVideos/Normal_Videos006_x264_10.png
```

Files with the same prefix before the final frame number are grouped into the same video clip.

Example:

```text
Arson042_x264_1170.png
Arson042_x264_1560.png
```

Both are assigned to:

```text
clip_id = Arson042_x264
```

The number after the final underscore is treated as the frame index, allowing frames to be sorted in temporal order.

### Label assignment

Binary labels are inferred from the folder name:

```text
NormalVideos folder → label 0
Any anomaly class folder → label 1
```

This notebook therefore creates video-level binary labels rather than frame-level anomaly labels.

### XD-Violence feature format

XD-Violence is loaded from precomputed I3D feature files:

```text
datasets/XD_Violence/i3d-features/RGB/
datasets/XD_Violence/i3d-features/Flow/
datasets/XD_Violence/i3d-features/RGBTest/
datasets/XD_Violence/i3d-features/FlowTest/
```

RGB features represent visual appearance, while Flow features represent motion. The notebook pairs RGB and Flow files and later fuses them into fixed-length feature bags.

## Final submitted configuration

The submitted project uses the following main configuration:

```text
dataset_mode = both
feature_mode = 3d
num_segments = 32
UCF feature type = R(2+1)D-18 clip features
XD feature type = RGB + Flow I3D fused features
final MIL input dimension = 512
CLIP semantic extension = enabled
CLIP semantic loss weight = 0.1
```

The final pipeline is a **feature-based spatiotemporal MIL pipeline**. The models are trained on extracted segment-level feature bags rather than directly on raw video frames.


## 0. Python environment and dependency check

This section prepares the Python environment required for the project. The notebook uses PyTorch, torchvision, pandas, NumPy, scikit-learn, OpenCV, matplotlib, tqdm, UMAP, and OpenCLIP.

The environment setup supports different execution platforms, including local machines, lab/HPC machines and Google Colab. The notebook checks the available hardware and selects the appropriate device:

- NVIDIA CUDA GPU
- Apple Silicon MPS
- CPU fallback

This improves reproducibility across different machines and allows the notebook to run even when GPU acceleration is unavailable.

After installing new packages, the kernel should be restarted before continuing so that the correct package versions are loaded.


In [1]:
# CUDA allocator (set before `import torch` in the dependency cell). Reduces fragmentation on some GPUs.
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


'expandable_segments:True'

In [2]:
# ============================================================================
# UNIVERSAL DEPENDENCY INSTALLER - FIXED VERSION
# Correctly installs ALL packages including torchvision
# ============================================================================

import importlib.util
import subprocess
import sys
import platform

# List of all required packages (including torch and torchvision)
_REQ = (
    "torch torchvision numpy pandas tqdm scikit-learn "
    "opencv-python-headless Pillow matplotlib umap-learn "
    "open-clip-torch"
).split()


def detect_hardware():
    """Detect available hardware and return the appropriate PyTorch variant"""
    system = platform.system()
    machine = platform.machine()
    
    print("=" * 60)
    print("SYSTEM DETECTION")
    print("=" * 60)
    print(f"Operating System: {system}")
    print(f"Architecture: {machine}")
    
    # Check for NVIDIA GPU
    has_nvidia = False
    nvidia_gpu_name = None
    cuda_driver_version = None
    
    try:
        result = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], 
                               capture_output=True, text=True, timeout=5)
        if result.returncode == 0 and result.stdout.strip():
            has_nvidia = True
            nvidia_gpu_name = result.stdout.strip().split('\n')[0]
            
            result2 = subprocess.run(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'],
                                    capture_output=True, text=True, timeout=5)
            if result2.returncode == 0:
                cuda_driver_version = result2.stdout.strip().split('\n')[0]
            
            print(f"✓ NVIDIA GPU detected: {nvidia_gpu_name}")
            print(f"✓ CUDA Driver: {cuda_driver_version}")
    except:
        pass
    
    # Check for Apple Silicon
    is_apple_silicon = (system == 'Darwin' and machine == 'arm64')
    if is_apple_silicon:
        print("✓ Apple Silicon (M1/M2/M3) detected")
    
    print("-" * 60)
    
    return {
        'has_nvidia': has_nvidia,
        'nvidia_gpu': nvidia_gpu_name,
        'cuda_driver': cuda_driver_version,
        'is_apple_silicon': is_apple_silicon,
        'system': system,
        'machine': machine
    }


def get_pytorch_install_cmd(hardware):
    """Return the appropriate PyTorch install command"""
    
    if hardware['has_nvidia']:
        cuda_driver = hardware['cuda_driver']
        cuda_version = "cu118"
        
        if cuda_driver:
            driver_major = int(cuda_driver.split('.')[0])
            if driver_major >= 12:
                cuda_version = "cu121"
                print(f"✓ Using CUDA 12.1 (driver {cuda_driver})")
            else:
                print(f"✓ Using CUDA 11.8 (driver {cuda_driver})")
        
        print(f"Installing PyTorch with CUDA {cuda_version}...")
        return [sys.executable, "-m", "pip", "install", "--upgrade",
                "torch", "torchvision", "torchaudio",
                "--index-url", f"https://download.pytorch.org/whl/{cuda_version}"]
    
    elif hardware['is_apple_silicon']:
        print("Installing PyTorch with MPS support...")
        return [sys.executable, "-m", "pip", "install", "--upgrade",
                "torch", "torchvision", "torchaudio"]
    
    else:
        print("Installing PyTorch CPU version...")
        return [sys.executable, "-m", "pip", "install", "--upgrade",
                "torch", "torchvision", "torchaudio",
                "--index-url", "https://download.pytorch.org/whl/cpu"]


def install_packages(hardware):
    """Install all required packages - FIXED: installs ALL packages together"""
    
    print("\n" + "=" * 60)
    print("INSTALLING ALL PACKAGES")
    print("=" * 60)
    
    # Install PyTorch, torchvision, torchaudio together
    install_cmd = get_pytorch_install_cmd(hardware)
    try:
        subprocess.check_call(install_cmd)
        print("✓ PyTorch + torchvision + torchaudio installed!")
    except subprocess.CalledProcessError as e:
        print(f"✗ PyTorch installation failed: {e}")
        print("Trying fallback CPU version...")
        fallback_cmd = [sys.executable, "-m", "pip", "install", "--upgrade",
                       "torch", "torchvision", "torchaudio",
                       "--index-url", "https://download.pytorch.org/whl/cpu"]
        subprocess.check_call(fallback_cmd)
    
    # Install remaining packages (numpy, pandas, etc.)
    remaining = [p for p in _REQ if p not in ["torch", "torchvision", "torchaudio"]]
    if remaining:
        print("\n" + "=" * 60)
        print("INSTALLING REMAINING PACKAGES")
        print("=" * 60)
        print(f"Installing: {' '.join(remaining)}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", *remaining])
        print("✓ All remaining packages installed!")
    
    print("=" * 60)


def verify_installation():
    """Verify everything works"""
    print("\n" + "=" * 60)
    print("VERIFYING INSTALLATION")
    print("=" * 60)
    
    # Check PyTorch and torchvision specifically
    try:
        import torch
        import torchvision
        print(f"✓ torch: {torch.__version__}")
        print(f"✓ torchvision: {torchvision.__version__}")
        
        if torch.cuda.is_available():
            print(f"✓ CUDA: True | GPU: {torch.cuda.get_device_name(0)}")
            print(f"✓ CUDA Version: {torch.version.cuda}")
            x = torch.randn(100, 100).cuda()
            print("✓ CUDA operation test passed")
        elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
            print("✓ MPS: True (Apple Silicon GPU)")
        else:
            print("✓ Device: CPU")
    except ImportError as e:
        print(f"✗ Import error: {e}")
        return False
    
    # Check other packages
    packages_to_check = [
        ('numpy', 'numpy'),
        ('pandas', 'pandas'),
        ('sklearn', 'scikit-learn'),
        ('cv2', 'opencv'),
        ('PIL', 'Pillow'),
        ('matplotlib', 'matplotlib'),
        ('umap', 'umap-learn'),
        ('tqdm', 'tqdm')
    ]
    
    print("\nOther packages:")
    all_ok = True
    for import_name, pkg_name in packages_to_check:
        try:
            if import_name == 'sklearn':
                import sklearn
                print(f"  ✓ {pkg_name}: {sklearn.__version__}")
            elif import_name == 'cv2':
                import cv2
                print(f"  ✓ {pkg_name}: {cv2.__version__}")
            elif import_name == 'PIL':
                from PIL import Image
                print(f"  ✓ {pkg_name}: {Image.__version__}")
            elif import_name == 'umap':
                import umap
                print(f"  ✓ {pkg_name}: {umap.__version__}")
            else:
                module = __import__(import_name)
                version = getattr(module, '__version__', 'unknown')
                print(f"  ✓ {pkg_name}: {version}")
        except ImportError:
            print(f"  ✗ {pkg_name}: NOT INSTALLED")
            all_ok = False
    
    print("=" * 60)
    if all_ok:
        print("✓ All systems ready!")
    else:
        print("⚠️  Some packages missing - run installer again")
    print("=" * 60)
    
    return all_ok


# ============================================================================
# MAIN EXECUTION
# ============================================================================

print("\n" + "=" * 60)
print("UNIVERSAL DEPENDENCY INSTALLER (FIXED)")
print("Detecting system and installing correct PyTorch version")
print("=" * 60 + "\n")

# Detect hardware
hardware = detect_hardware()

# Install ALL packages (torch, torchvision, etc. together)
install_packages(hardware)

# Verify installation
verify_installation()

print("\n⚠️  IMPORTANT: If you see import errors, restart the kernel:")
print("   Kernel → Restart Kernel (Jupyter)")
print("   or Ctrl+Shift+P → 'Python: Restart Kernel' (VS Code)")


UNIVERSAL DEPENDENCY INSTALLER (FIXED)
Detecting system and installing correct PyTorch version

SYSTEM DETECTION
Operating System: Linux
Architecture: x86_64
✓ NVIDIA GPU detected: NVIDIA RTX 4000 Ada Generation
✓ CUDA Driver: 580.126.09
------------------------------------------------------------

INSTALLING ALL PACKAGES
✓ Using CUDA 12.1 (driver 580.126.09)
Installing PyTorch with CUDA cu121...
Looking in indexes: https://download.pytorch.org/whl/cu121
✓ PyTorch + torchvision + torchaudio installed!

INSTALLING REMAINING PACKAGES
Installing: numpy pandas tqdm scikit-learn opencv-python-headless Pillow matplotlib umap-learn open-clip-torch
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 36.3 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 26.7 MB/s  0:00:00eta 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.4
    Uninstalling numpy-2.4.4:
      Successfully uninstalled numpy-2.4.4
  Attempting uninstall: pa


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


✓ All remaining packages installed!

VERIFYING INSTALLATION
✓ torch: 2.5.1+cu121
✓ torchvision: 0.20.1+cu121
✓ CUDA: True | GPU: NVIDIA RTX 4000 Ada Generation
✓ CUDA Version: 12.1
✓ CUDA operation test passed

Other packages:
  ✓ numpy: 2.4.6
  ✓ pandas: 3.0.3
  ✓ scikit-learn: 1.8.0
  ✓ opencv: 4.13.0
  ✓ Pillow: 12.2.0
  ✓ matplotlib: 3.10.9


I0000 00:00:1780072585.735680  606904 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780072585.761611  606904 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780072586.619321  606904 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


  ✓ umap-learn: 0.5.12
  ✓ tqdm: 4.67.3
✓ All systems ready!

⚠️  IMPORTANT: If you see import errors, restart the kernel:
   Kernel → Restart Kernel (Jupyter)
   or Ctrl+Shift+P → 'Python: Restart Kernel' (VS Code)


## 1. Google Drive / local data mode

This notebook can run in either a local/HPC environment or a Google Colab environment.

1. **Local or HPC mode**  
   Data is read from the local project folders, such as `/scratch/VAD` or the local `datasets/` directory.

2. **Google Colab mode**  
   Data can be read from Google Drive when `USE_GOOGLE_DRIVE = True`.

The submitted run uses the local/HPC-style configuration, with project data stored under `/scratch/VAD`.

Before continuing, the expected dataset and output folders should exist:

```text
UCF-Crime frame data
XD-Violence I3D feature data
processed output folder
artifacts folder
```

The configuration cell prints the resolved paths and confirms whether the required directories are available.


### 1b. XD-Violence I3D feature layout

XD-Violence is used through pre-extracted I3D features rather than raw video frames.

Expected folder layout:

```text
datasets/XD_Violence/i3d-features/
    RGB/
    Flow/
    RGBTest/
    FlowTest/
```

The notebook pairs RGB and Flow feature files belonging to the same clip.

- `RGB` and `Flow` contain training features.
- `RGBTest` and `FlowTest` contain test features.
- Each `.npy` file contains temporal snippet-level features.

### Label inference

Binary labels are inferred from the filename:

```text
filename contains label_A → normal video → label 0
otherwise → anomalous/violent video → label 1
```

### RGB and Flow features

RGB features capture appearance information such as people, objects and scene context. Optical-flow features capture motion patterns such as fighting, vehicle movement, explosions or rapid temporal changes.

Using both RGB and Flow provides a stronger spatiotemporal representation than using RGB alone.


### 1c. UCF-Crime frame data layout

UCF-Crime is stored as extracted PNG frames. The configuration variable `DATA_ROOT` points to the UCF-Crime dataset folder.

Expected structure:

```text
DATA_ROOT/
    Train/
        NormalVideos/
        Abuse/
        Arrest/
        Arson/
        Assault/
        Burglary/
        Explosion/
        Fighting/
        RoadAccidents/
        Robbery/
        Shooting/
        Shoplifting/
        Stealing/
        Vandalism/
    Test/
        NormalVideos/
        anomaly class folders...
```

The notebook scans all `.png` files recursively, groups frames into clips, and assigns binary labels.

### Label rule

```text
NormalVideos → label 0
All other class folders → label 1
```

### Split handling

UCF-Crime already provides an official Train/Test organisation. The official test set is kept unchanged, while the official training set is further split into training and validation subsets. This prevents test leakage and provides a validation set for model selection.


## 2. Configuration

This section defines the main project settings used by all later notebooks.

The configuration controls:

- dataset mode: UCF only, XD only, or both
- feature mode: 2D, 3D, or fusion
- number of temporal segments per video
- feature dimensions
- train/validation/test split settings
- output directories
- run ID and result folders
- GPU/CPU device selection
- CLIP semantic extension settings

### Final submitted configuration

The submitted run uses:

```text
DATASET_MODE = both
feature_mode = 3d
NUM_SEGMENTS = 32
USE_3D_FEATURES = True
USE_FUSION = False
MIL_INPUT_DIM = 512
CLIP semantic extension = True
CLIP semantic loss weight = 0.1
CLIP text encoder trainable = False
```

### Number of temporal segments

Each video is converted into a fixed-length bag of 32 temporal segments. This makes batching consistent and follows the common MIL setup used in weakly supervised video anomaly detection.

### Feature mode

The final run uses spatiotemporal features because video anomaly detection depends heavily on both appearance and motion.

```text
UCF-Crime → R(2+1)D-18 features
XD-Violence → RGB + Flow I3D fused features
```

A purely 2D image representation can capture static appearance but does not fully represent temporal movement. The 3D feature configuration is therefore used for the final submitted experiments.

### CLIP semantic extension

The CLIP semantic settings are included in the shared configuration because later notebooks use these values when enabling the extra semantic extension.

The actual submitted configuration uses:

```text
CLIP_SEMANTIC_LOSS_WEIGHT = 0.1
```

The run folder name may contain an older tag with `ClipWeight_0.05`, but the actual configuration used for the submitted model is `0.1`.


In [6]:
# ============================================================================
# DYNAMIC ENVIRONMENT CONFIGURATION
# Strong spatiotemporal default:
# - UCF-Crime: raw frame clips -> R(2+1)D-18 clip encoder
# - XD-Violence: paired RGB + Flow I3D features -> fused bags
# Automatically detects local vs Colab and sets up paths accordingly
# ============================================================================

import sys
import platform
from pathlib import Path
import torch
import json
import hashlib
from datetime import datetime
import os

print("=" * 70)
print("ENVIRONMENT DETECTION & CONFIGURATION")
print("=" * 70)

# ============================================================================
# 1. ENVIRONMENT DETECTION
# ============================================================================

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

# Detect OS
SYSTEM = platform.system()
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Operating System: {SYSTEM}")
print(f"Python: {sys.version.split()[0]}")

# ============================================================================
# 2. PATH CONFIGURATION (Auto-detects based on environment)
# ============================================================================

if IN_COLAB:
    # Google Colab paths
    print("\n" + "-" * 70)
    print("CONFIGURING COLAB PATHS")
    print("-" * 70)

    # Mount Google Drive if not already mounted
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("✓ Google Drive mounted")
    except ImportError:
        print("⚠️  Not in Colab environment")

    # Colab dataset paths
    PROJECT_ROOT = Path("/content/drive/MyDrive/vad_project")
    DATA_ROOT = str(PROJECT_ROOT / "datasets" / "UCF_Crime")
    XD_VIOLENCE_ROOT = str(PROJECT_ROOT / "datasets" / "XD_Violence")
    I3D_FEATURES_DIR = PROJECT_ROOT / "datasets" / "XD_Violence" / "i3d-features"
    PROCESSED_ROOT = str(PROJECT_ROOT / "processed")

    USE_GOOGLE_DRIVE = True
    USE_LOCAL_COPY = False
    VAD_PROJECT_ROOT = PROJECT_ROOT

else:
    # Local: layout per VAD_FOLDER_TREE / README — <root>/datasets/... and <root>/experiments/runs.
    # Optional: export VAD_PROJECT_ROOT=<absolute path to parent of datasets/>
    print("\n" + "-" * 70)
    print("CONFIGURING LOCAL PATHS")
    print("-" * 70)

    import os

    def _resolve_vad_project_root() -> Path:
        env = os.environ.get("VAD_PROJECT_ROOT")
        if env:
            return Path(env).expanduser().resolve()
        cwd = Path.cwd().resolve()
        for p in [cwd, *list(cwd.parents)[:12]]:
            for ucf_name in ("ucf-crime-dataset", "UCF_Crime"):
                cand = p / "datasets" / ucf_name
                if cand.is_dir():
                    return p
        return cwd

    #VAD_PROJECT_ROOT = _resolve_vad_project_root()
    VAD_PROJECT_ROOT = Path(r"/scratch/VAD")
    print(f"VAD_PROJECT_ROOT: {VAD_PROJECT_ROOT}")
    if os.environ.get("VAD_PROJECT_ROOT"):
        print("(using environment variable VAD_PROJECT_ROOT)")

    _ucf_opts = [
        VAD_PROJECT_ROOT / "datasets" / "ucf-crime-dataset",
        VAD_PROJECT_ROOT / "datasets" / "UCF_Crime",
    ]
    _ucf_dir = next((p for p in _ucf_opts if p.is_dir()), _ucf_opts[0])
    DATA_ROOT = str(_ucf_dir)
    XD_VIOLENCE_ROOT = str(VAD_PROJECT_ROOT / "datasets" / "XD_Violence")
    I3D_FEATURES_DIR = VAD_PROJECT_ROOT / "datasets" / "XD_Violence" / "i3d-features"
    PROCESSED_ROOT = str(VAD_PROJECT_ROOT / "processed")

    USE_GOOGLE_DRIVE = False
    USE_LOCAL_COPY = False


# ============================================================================
# 3. EXPERIMENT-SPECIFIC PATHS AND RUN CONFIGURATION
# ============================================================================

BASE_RUNS_DIR = VAD_PROJECT_ROOT / "experiments" / "runs"

USE_RUNS_DIR = True
RUN_TAG = "BEST_Final_Run_Zain_ClipWeight_0.05"

# Single source of truth for experiment knobs
CFG = {
    "dataset_mode": "both",       # "ucf" | "xd" | "both"
    "feature_mode": "3d",         # "2d" | "3d" | "fusion"
    "num_segments": 32,
    "seed": 42,
    "lr": 1e-5,
    "batch_size": 4,
    "max_epochs": 40,
    "patience": 10,
}
CFG["topk_k"] = max(1, int(CFG["num_segments"]) // 8)

# ============================================================================
# OPTIONAL EXTRA-CREDIT SWITCHES
# ============================================================================


# Extra credit switch.
# ============================================================================
# INTEGRATED CLIP SEMANTIC HEAD SETTINGS
# ============================================================================
USE_INTEGRATED_CLIP_SEMANTIC_HEAD = True

USE_CLIP_TEXT_EXTRA_CREDIT = True

# Weight for semantic text-label loss during MIL training.
# Keep small so anomaly detection remains the main task.
CLIP_SEMANTIC_LOSS_WEIGHT = 0.1

# CLIP text encoder configuration.
CLIP_MODEL_NAME = "ViT-B-32"
CLIP_PRETRAINED = "openai"

# Safer setting: freeze CLIP text encoder and train only the MIL projection head.
# This still integrates CLIP semantic space into training.
CLIP_TEXT_ENCODER_TRAINABLE = False

# Text labels used by the CLIP text encoder in Notebook 5.
# These are semantic anomaly labels, not necessarily supervised multiclass labels.
# Binary AUC/AP remains the main evaluation.

ANOMALY_LABEL_SET = [
    "normal",
    "abuse",
    "arrest",
    "arson",
    "assault",
    "burglary",
    "explosion",
    "fighting",
    "road accident",
    "robbery",
    "shooting",
    "shoplifting",
    "stealing",
    "vandalism",
    "violence",
]

# Validate config
if CFG["feature_mode"] not in ("2d", "3d", "fusion"):
    raise ValueError("CFG['feature_mode'] must be '2d', '3d', or 'fusion'.")

USE_FUSION = CFG["feature_mode"] == "fusion"
USE_3D_FEATURES = CFG["feature_mode"] == "3d"

# Expose commonly used globals
SEED = int(CFG["seed"])
NUM_SEGMENTS = int(CFG["num_segments"])

# Build run-specific folder
if USE_RUNS_DIR:
    _cfg_for_hash = {
        "run_tag": RUN_TAG,
        **CFG,
    }
    _cfg_str = json.dumps(_cfg_for_hash, sort_keys=True)
    _cfg_hash = hashlib.sha1(_cfg_str.encode("utf-8")).hexdigest()[:8]
    _timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    RUN_ID = (
        f"vad_{RUN_TAG}_seg{NUM_SEGMENTS}_{CFG['dataset_mode']}_{CFG['feature_mode']}"
        f"_topkK{CFG['topk_k']}_lr{CFG['lr']}_bs{CFG['batch_size']}"
        f"_seed{SEED}_ep{CFG['max_epochs']}_h{_cfg_hash}_{_timestamp}"
    )

    PROJECT_ROOT = BASE_RUNS_DIR / RUN_ID
    PROCESSED_ROOT = str(PROJECT_ROOT / "processed")
    print(f"Run ID: {RUN_ID}")
    print(f"Project root (runs dir): {PROJECT_ROOT}")
else:
    PROJECT_ROOT = VAD_PROJECT_ROOT / "experiments" / "exp_32seg"
    PROCESSED_ROOT = str(PROJECT_ROOT / "processed")
    print(f"Project root (fixed): {PROJECT_ROOT}")

print(f"PROCESSED_ROOT: {PROCESSED_ROOT}")

# Save run config
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_CONFIG_EXPORT = {
    "use_integrated_clip_semantic_head": USE_INTEGRATED_CLIP_SEMANTIC_HEAD,
    "clip_semantic_loss_weight": CLIP_SEMANTIC_LOSS_WEIGHT,
    "clip_model_name": CLIP_MODEL_NAME,
    "clip_pretrained": CLIP_PRETRAINED,
    "clip_text_encoder_trainable": CLIP_TEXT_ENCODER_TRAINABLE,
    "run_tag": RUN_TAG,
    **CFG,
    "use_clip_text_extra_credit": USE_CLIP_TEXT_EXTRA_CREDIT,
    "anomaly_label_set": ANOMALY_LABEL_SET,
}

(PROJECT_ROOT / "run_config.json").write_text(
    json.dumps(RUN_CONFIG_EXPORT, indent=2, sort_keys=True),
    encoding="utf-8",
)
print(f"Wrote run_config.json to: {PROJECT_ROOT}")

# ============================================================================
# 4. CREATE DIRECTORY STRUCTURE
# ============================================================================

# EXPERIMENT_PLAYBOOK_Z: stable metadata + splits across RUN_ID; features stay per-run under PROCESSED_ROOT.
_PLAYBOOK_ARTIFACTS = Path(VAD_PROJECT_ROOT) / "artifacts"
_PLAYBOOK_ARTIFACTS.mkdir(parents=True, exist_ok=True)
METADATA_DIR = _PLAYBOOK_ARTIFACTS / "metadata"
SPLITS_DIR = _PLAYBOOK_ARTIFACTS / "splits"
FEATURES_DIR = Path(PROCESSED_ROOT) / "features"
FEATURES_DIR_XD = Path(PROCESSED_ROOT) / "features_xd"
FEATURES_3D_DIR = Path(PROCESSED_ROOT) / "features_3d"
FEATURES_FUSION_DIR = Path(PROCESSED_ROOT) / "features_fusion"
RESULTS_DIR = Path(PROCESSED_ROOT).resolve().parent / "results"

for d in [METADATA_DIR, SPLITS_DIR, FEATURES_DIR, FEATURES_DIR_XD,
          FEATURES_3D_DIR, FEATURES_FUSION_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def ucf_feature_dir() -> Path:
    """Return feature directory for UCF depending on feature mode."""
    if USE_FUSION:
        return FEATURES_FUSION_DIR
    return FEATURES_3D_DIR if USE_3D_FEATURES else FEATURES_DIR

print(f"\n✓ Output directories created under: {PROCESSED_ROOT}")

# ============================================================================
# 5. DATASET CONFIGURATION
# ============================================================================

print("\n" + "-" * 70)
print("DATASET CONFIGURATION")
print("-" * 70)

DATASET_MODE = CFG["dataset_mode"]
print(f"Dataset mode: {DATASET_MODE}")

# Check UCF dataset
if Path(DATA_ROOT).exists():
    ucf_frames = list(Path(DATA_ROOT).rglob("*.png"))
    print(f"✓ UCF dataset: {len(ucf_frames)} frames found")
else:
    print(f"⚠️  UCF dataset not found at: {DATA_ROOT}")

# Check XD dataset
if I3D_FEATURES_DIR.exists():
    xd_files = list(I3D_FEATURES_DIR.rglob("*.npy"))
    print(f"✓ XD dataset: {len(xd_files)} feature files found")
else:
    print(f"⚠️  XD dataset not found at: {I3D_FEATURES_DIR}")

# ============================================================================
# 6. MODEL HYPERPARAMETERS
# ============================================================================

print("\n" + "-" * 70)
print("MODEL HYPERPARAMETERS")
print("-" * 70)

# Core parameters
FRAMES_PER_SEGMENT = 2
RESIZE_HW = (224, 224)
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.8, 0.1, 0.1

# Feature dimensions
FEATURE_DIM = 2048
XD_FEATURE_DIM = 1024
XD_SEGMENTS_RAW = 360

# 3D feature extraction parameters
FEATURE_DIM_3D = 512
CLIP_LEN = 16
CLIP_STRIDE = 8

# XD stream projection size in 3D mode (RGB + Flow => 256 + 256)
XD_STREAM_OUT_DIM = FEATURE_DIM_3D // 2

print(f"SEED: {SEED}")
print(f"NUM_SEGMENTS: {NUM_SEGMENTS}")
print(f"FEATURE_DIM: {FEATURE_DIM}")
print(f"feature_mode:     {CFG['feature_mode']}")
print(f"USE_3D_FEATURES: {USE_3D_FEATURES}")
print(f"USE_FUSION:      {USE_FUSION}")

if USE_FUSION:
    print(f"  → Early fusion: ResNet (2D) + R(2+1)D (3D) -> {FEATURE_DIM + FEATURE_DIM_3D}-dim bags")
elif USE_3D_FEATURES:
    print(f"  → Using R(2+1)D-18 features for UCF and RGB+Flow fused I3D bags for XD")
else:
    print(f"  → Using ResNet50 features (spatial)")

PHASE_ACTIVE = 2

# ============================================================================
# 7. DEVICE DETECTION (with fallback)
# ============================================================================

print("\n" + "-" * 70)
print("DEVICE DETECTION")
print("-" * 70)

FORCE_CPU = False

def select_torch_device(force_cpu: bool = False) -> torch.device:
    """Intelligently select CUDA/CPU device with fallback."""
    if force_cpu:
        print("TORCH_DEVICE: cpu (FORCE_CPU=True)")
        return torch.device("cpu")

    if not torch.cuda.is_available():
        print("TORCH_DEVICE: cpu (CUDA not available)")
        return torch.device("cpu")

    try:
        x = torch.randn(1, 3, 224, 224, device="cuda")
        w = torch.randn(64, 3, 7, 7, device="cuda")
        _ = torch.nn.functional.conv2d(x, w, padding=3)
        del _, w, x
        torch.cuda.empty_cache()

        gpu_name = torch.cuda.get_device_name(0)
        cuda_version = torch.version.cuda
        print(f"✓ CUDA available: {gpu_name}")
        print(f"✓ CUDA version: {cuda_version}")
        print("TORCH_DEVICE: cuda (smoke test passed)")
        return torch.device("cuda")

    except Exception as e:
        print(f"⚠️  CUDA reported but ops failed: {str(e)[:100]}...")
        print("TORCH_DEVICE: cpu (fallback)")
        return torch.device("cpu")

TORCH_DEVICE = select_torch_device(FORCE_CPU)
device = TORCH_DEVICE

# ============================================================================
# 8. MIL CONFIGURATION
# ============================================================================

if USE_FUSION:
    MIL_INPUT_DIM = FEATURE_DIM + FEATURE_DIM_3D
elif USE_3D_FEATURES:
    MIL_INPUT_DIM = FEATURE_DIM_3D
else:
    MIL_INPUT_DIM = FEATURE_DIM

MIL_DUMMY_T = 32 if (USE_3D_FEATURES or USE_FUSION) else NUM_SEGMENTS

# ============================================================================
# 9. VERIFICATION SUMMARY
# ============================================================================

print("\n" + "=" * 70)
print("CONFIGURATION SUMMARY")
print("=" * 70)
print(f"Environment:      {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Dataset mode:     {DATASET_MODE}")
print(f"Data root:        {DATA_ROOT}")
print(f"XD root:          {XD_VIOLENCE_ROOT}")
print(f"Processed root:   {PROCESSED_ROOT}")
print(f"Results root:     {RESULTS_DIR}")
print(f"Device:           {TORCH_DEVICE}")
print(f"Features:         {'Fusion' if USE_FUSION else ('R(2+1)D-18 + XD RGB/Flow' if USE_3D_FEATURES else 'ResNet50')}")
print(f"Segments/video:   {NUM_SEGMENTS}")
print(f"Input feature dim:{MIL_INPUT_DIM}")
print(f"CLIP extra credit:{USE_CLIP_TEXT_EXTRA_CREDIT}")
print(f"Text labels:      {len(ANOMALY_LABEL_SET)} labels")
print("=" * 70)

print("\nDataset status:")
print(f"  UCF Crime:  {'✓ Found' if Path(DATA_ROOT).exists() else '✗ Missing'}")
print(f"  XD Violence:{'✓ Found' if Path(I3D_FEATURES_DIR).exists() else '✗ Missing'}")
print(f"  I3D Features:{'✓ Found' if Path(I3D_FEATURES_DIR).exists() else '✗ Missing'}")

print("\n⚠️  IMPORTANT: If you changed DATASET_MODE or feature_mode, restart kernel and re-run this cell.")

ENVIRONMENT DETECTION & CONFIGURATION
Environment: Local
Operating System: Linux
Python: 3.12.3

----------------------------------------------------------------------
CONFIGURING LOCAL PATHS
----------------------------------------------------------------------
VAD_PROJECT_ROOT: /scratch/VAD
Run ID: vad_BEST_Final_Run_Zain_ClipWeight_0.05_seg32_both_3d_topkK4_lr1e-05_bs4_seed42_ep40_hc2f41927_20260529_174217
Project root (runs dir): /scratch/VAD/experiments/runs/vad_BEST_Final_Run_Zain_ClipWeight_0.05_seg32_both_3d_topkK4_lr1e-05_bs4_seed42_ep40_hc2f41927_20260529_174217
PROCESSED_ROOT: /scratch/VAD/experiments/runs/vad_BEST_Final_Run_Zain_ClipWeight_0.05_seg32_both_3d_topkK4_lr1e-05_bs4_seed42_ep40_hc2f41927_20260529_174217/processed
Wrote run_config.json to: /scratch/VAD/experiments/runs/vad_BEST_Final_Run_Zain_ClipWeight_0.05_seg32_both_3d_topkK4_lr1e-05_bs4_seed42_ep40_hc2f41927_20260529_174217

✓ Output directories created under: /scratch/VAD/experiments/runs/vad_BEST_Final_Run_Z

In [7]:
# ============================================================================
# EXPORT EXTRA-CREDIT CONFIG FLAGS
# ============================================================================
# This small file is useful for later notebooks and for the final report.
# It records whether the CLIP extra credit section is enabled.

extra_config = {
    "use_clip_text_extra_credit": bool(USE_CLIP_TEXT_EXTRA_CREDIT),
    "use_integrated_clip_semantic_head": USE_INTEGRATED_CLIP_SEMANTIC_HEAD,
    "clip_semantic_loss_weight": CLIP_SEMANTIC_LOSS_WEIGHT,
    "clip_model_name": CLIP_MODEL_NAME,
    "clip_pretrained": CLIP_PRETRAINED,
    "clip_text_encoder_trainable": CLIP_TEXT_ENCODER_TRAINABLE,
    "anomaly_label_set": ANOMALY_LABEL_SET,
    "num_anomaly_text_labels": len(ANOMALY_LABEL_SET),
    "spatiotemporal_mode_note": (
        "Feature-based spatiotemporal MIL: UCF uses R(2+1)D clip features "
        "and XD uses fused RGB/Flow I3D bags. This is not raw end-to-end video training."
        if USE_3D_FEATURES
        else "Spatial/fusion configuration depending on feature_mode."
    ),
}

extra_config_path = Path(PROJECT_ROOT) / "model_extra_config.json"
extra_config_path.write_text(
    json.dumps(extra_config, indent=2, sort_keys=True),
    encoding="utf-8",
)

print("Wrote:", extra_config_path)
display(extra_config)

Wrote: /scratch/VAD/experiments/runs/vad_BEST_Final_Run_Zain_ClipWeight_0.05_seg32_both_3d_topkK4_lr1e-05_bs4_seed42_ep40_hc2f41927_20260529_174217/model_extra_config.json


{'use_clip_text_extra_credit': True,
 'use_integrated_clip_semantic_head': True,
 'clip_semantic_loss_weight': 0.1,
 'clip_model_name': 'ViT-B-32',
 'clip_pretrained': 'openai',
 'clip_text_encoder_trainable': False,
 'anomaly_label_set': ['normal',
  'abuse',
  'arrest',
  'arson',
  'assault',
  'burglary',
  'explosion',
  'fighting',
  'road accident',
  'robbery',
  'shooting',
  'shoplifting',
  'stealing',
  'vandalism',
  'violence'],
 'num_anomaly_text_labels': 15,
 'spatiotemporal_mode_note': 'Feature-based spatiotemporal MIL: UCF uses R(2+1)D clip features and XD uses fused RGB/Flow I3D bags. This is not raw end-to-end video training.'}

## 3. XD-Violence metadata builder

This section builds metadata for XD-Violence from precomputed I3D feature files.

The code scans:

```text
RGB/
Flow/
RGBTest/
FlowTest/
```

It pairs RGB and Flow feature files belonging to the same clip and produces:

```text
xd_metadata_df
xd_train_df
xd_val_df
xd_test_df
```

### Metadata row structure

Each metadata row represents one XD-Violence clip/feature pair and contains:

```text
clip_id
rgb_path
flow_path
label
split
dataset
```

### Split logic

The split is inferred from the feature folder:

```text
RGB and Flow → train
RGBTest and FlowTest → test
```

The official XD training portion is further split into training and validation subsets:

```text
XD train rows → 80% train / 20% validation
XD test rows → kept as test
```

In the submitted run, this produced:

```text
XD train = 15816
XD validation = 3954
XD test = 4000
```

This metadata is later used to construct fixed-length MIL feature bags for training and evaluation.


In [8]:
# ============================================================================
# BUILD XD-VIOLENCE PAIRED METADATA
# Pairs RGB and Flow feature files by relative path
# Infers split and label from filenames and folder structure
# Produces:
# - xd_metadata_df
# - xd_train_df
# - xd_val_df
# - xd_test_df
# ============================================================================

from pathlib import Path
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split

print("=" * 70)
print("BUILD XD-VIOLENCE PAIRED METADATA")
print("=" * 70)

# ============================================================================
# 1. HELPER FUNCTIONS
# ============================================================================

def _norm_rel_without_root(p: Path, root_name: str) -> str:
    """Remove top-level modality folder name from relative path."""
    rel = p.as_posix()
    prefix = root_name + "/"
    if rel.startswith(prefix):
        return rel[len(prefix):]
    return rel

def build_xd_metadata(i3d_dir: Path) -> pd.DataFrame:
    """
    Build paired XD metadata from RGB / Flow train and test folders.

    Expected structure:
    - RGB
    - Flow
    - RGBTest
    - FlowTest
    """
    i3d_dir = Path(i3d_dir)

    roots = {
        "train_rgb": i3d_dir / "RGB",
        "train_flow": i3d_dir / "Flow",
        "test_rgb": i3d_dir / "RGBTest",
        "test_flow": i3d_dir / "FlowTest",
    }

    paired = {}

    for split_name, root in roots.items():
        if not root.exists():
            print(f"⚠️  Missing XD root: {root}")
            continue

        modality = "rgb" if "rgb" in split_name else "flow"
        split = "train" if split_name.startswith("train") else "test"

        for p in root.rglob("*.npy"):
            rel = _norm_rel_without_root(p.relative_to(i3d_dir), root.name)
            rec = paired.setdefault(
                (split, rel),
                {
                    "split": split,
                    "rel": rel,
                    "rgb_path": None,
                    "flow_path": None,
                }
            )
            rec[f"{modality}_path"] = str(p.resolve())

    rows = []
    for (_, rel), rec in sorted(paired.items(), key=lambda x: x[0][1]):
        rgb_path = rec["rgb_path"]
        flow_path = rec["flow_path"]

        if rgb_path is None and flow_path is None:
            continue

        basename = Path(rgb_path or flow_path).name
        label = 0 if "label_A" in basename else 1
        clip_id = Path(rel).with_suffix("").as_posix().replace("/", "__")

        rows.append({
            "clip_id": clip_id,
            "rgb_path": rgb_path,
            "flow_path": flow_path,
            "label": label,
            "split": rec["split"],
            "dataset": "xd",
        })

    cols = ["clip_id", "rgb_path", "flow_path", "label", "split", "dataset"]
    return pd.DataFrame(rows, columns=cols)

# ============================================================================
# 2. LOAD OR BUILD CACHE
# ============================================================================

_xd_cache = METADATA_DIR / "xd_paired_metadata.pkl"
_use_xd_cache = False

if _xd_cache.exists():
    try:
        with open(_xd_cache, "rb") as f:
            _data = pickle.load(f)
        if isinstance(_data, dict) and "records" in _data:
            xd_metadata_df = pd.DataFrame(_data["records"])
            _use_xd_cache = True
            print("Loaded", len(xd_metadata_df), "XD paired rows from cache.")
    except Exception as e:
        print("XD cache load failed:", e)

if not _use_xd_cache:
    xd_metadata_df = build_xd_metadata(I3D_FEATURES_DIR)
    METADATA_DIR.mkdir(parents=True, exist_ok=True)

    with open(_xd_cache, "wb") as f:
        pickle.dump({"records": xd_metadata_df.to_dict("records")}, f)

    print("XD metadata:", len(xd_metadata_df), "rows (saved to cache).")

# ============================================================================
# 3. VALIDATION CHECKS
# ============================================================================

required_cols = {"clip_id", "rgb_path", "flow_path", "label", "split", "dataset"}
missing_cols = required_cols - set(xd_metadata_df.columns)

if missing_cols:
    raise KeyError(f"xd_metadata_df missing columns: {missing_cols}")

print("Split counts:", xd_metadata_df["split"].value_counts().to_dict())
print("Label counts:", xd_metadata_df["label"].value_counts().to_dict())

print("Available modalities:", {
    "rgb_only": int(((xd_metadata_df["rgb_path"].notna()) & (xd_metadata_df["flow_path"].isna())).sum()),
    "flow_only": int(((xd_metadata_df["rgb_path"].isna()) & (xd_metadata_df["flow_path"].notna())).sum()),
    "rgb_flow": int(((xd_metadata_df["rgb_path"].notna()) & (xd_metadata_df["flow_path"].notna())).sum()),
})

# ============================================================================
# 4. TRAIN / VAL / TEST SPLIT
# ============================================================================

xd_train_full = xd_metadata_df[xd_metadata_df["split"] == "train"].copy()
xd_test_df = xd_metadata_df[xd_metadata_df["split"] == "test"].copy()

if len(xd_train_full) > 0:
    xd_train_df, xd_val_df = train_test_split(
        xd_train_full,
        test_size=0.2,
        stratify=xd_train_full["label"],
        random_state=SEED
    )
    xd_train_df = xd_train_df.reset_index(drop=True)
    xd_val_df = xd_val_df.reset_index(drop=True)

    print("XD 80:20 split: train", len(xd_train_df), "| val", len(xd_val_df), "| test", len(xd_test_df))
else:
    xd_train_df = pd.DataFrame(columns=xd_metadata_df.columns)
    xd_val_df = pd.DataFrame(columns=xd_metadata_df.columns)
    print("XD: no train rows; train/val empty. Test:", len(xd_test_df))

# ============================================================================
# 5. PREVIEW
# ============================================================================

display(xd_metadata_df.head())

BUILD XD-VIOLENCE PAIRED METADATA
Loaded 23770 XD paired rows from cache.
Split counts: {'train': 19770, 'test': 4000}
Label counts: {1: 12025, 0: 11745}
Available modalities: {'rgb_only': 0, 'flow_only': 0, 'rgb_flow': 23770}
XD 80:20 split: train 15816 | val 3954 | test 4000


,clip_id,rgb_path,flow_path,label,split,dataset
0,A.Beautiful.Mind.2001__#00-01-45_00-02-50_labe...,/scratch/VAD/datasets/XD_Violence/i3d-features...,/scratch/VAD/datasets/XD_Violence/i3d-features...,0,train,xd
1,A.Beautiful.Mind.2001__#00-01-45_00-02-50_labe...,/scratch/VAD/datasets/XD_Violence/i3d-features...,/scratch/VAD/datasets/XD_Violence/i3d-features...,0,train,xd
2,A.Beautiful.Mind.2001__#00-01-45_00-02-50_labe...,/scratch/VAD/datasets/XD_Violence/i3d-features...,/scratch/VAD/datasets/XD_Violence/i3d-features...,0,train,xd
3,A.Beautiful.Mind.2001__#00-01-45_00-02-50_labe...,/scratch/VAD/datasets/XD_Violence/i3d-features...,/scratch/VAD/datasets/XD_Violence/i3d-features...,0,train,xd
4,A.Beautiful.Mind.2001__#00-01-45_00-02-50_labe...,/scratch/VAD/datasets/XD_Violence/i3d-features...,/scratch/VAD/datasets/XD_Violence/i3d-features...,0,train,xd


## 4. XD feature bag creation

This section converts XD-Violence RGB and Flow I3D features into fixed-length MIL bags.

Each video is converted into:

```text
NUM_SEGMENTS × MIL_INPUT_DIM
```

For the submitted run:

```text
32 × 512
```

### Processing steps

1. Load RGB I3D features.
2. Load Flow I3D features.
3. Uniformly sample each stream to 32 temporal segments.
4. Align each stream to half of the final feature dimension.
5. Concatenate RGB and Flow features.

For the final 3D configuration:

```text
RGB stream → 256 dimensions
Flow stream → 256 dimensions
RGB + Flow → 512 dimensions
```

The dimensional alignment is performed using simple crop-or-pad logic rather than a learned projection. This ensures that XD feature bags match the same 512-dimensional input format used by the MIL models.

This step is required because the later models expect every video bag to have the same shape.


In [9]:
# ============================================================================
# BUILD XD RGB + FLOW FUSED BAGS
# Creates per-video fixed-length bags by:
# - loading paired RGB / Flow I3D features
# - temporal subsampling to NUM_SEGMENTS
# - projecting each stream to half-dimension
# - concatenating RGB + Flow
# Saves:
# - FEATURES_DIR_XD / <clip_id>.npy
# ============================================================================

import numpy as np
from pathlib import Path
from tqdm import tqdm

print("=" * 70)
print("BUILD XD RGB + FLOW FUSED BAGS")
print("=" * 70)

# ============================================================================
# 1. SETTINGS
# ============================================================================

XD_FEATURE_CAP_PER_CLASS = None  # Set an integer if you want to cap per class

# Output dimension depends on feature mode
_total_xd_out_dim = int(FEATURE_DIM_3D if USE_3D_FEATURES else FEATURE_DIM)
_stream_out_dim = int(_total_xd_out_dim // 2)

print(f"Total XD output dim: {_total_xd_out_dim}")
print(f"Per-stream output dim: {_stream_out_dim}")

# ============================================================================
# 2. HELPER FUNCTIONS
# ============================================================================

def temporal_subsample(raw: np.ndarray, num_segments: int) -> np.ndarray:
    """Uniformly subsample raw temporal features to fixed number of segments."""
    T = raw.shape[0]
    if T == 0:
        return np.zeros((num_segments, raw.shape[1]), dtype=np.float32)
    idx = np.linspace(0, T - 1, num_segments, dtype=int)
    return raw[idx].astype(np.float32)

def project_stream(raw: np.ndarray, out_dim: int) -> np.ndarray:
    """Simple crop-or-pad projection to fixed dimension."""
    in_dim = raw.shape[1]

    if in_dim == out_dim:
        return raw.astype(np.float32)

    if in_dim > out_dim:
        return raw[:, :out_dim].astype(np.float32)

    pad = np.zeros((raw.shape[0], out_dim - in_dim), dtype=np.float32)
    return np.concatenate([raw, pad], axis=1).astype(np.float32)

def load_stream(path_str):
    """Load a single RGB or Flow npy stream."""
    if path_str is None or (isinstance(path_str, float) and np.isnan(path_str)):
        return None

    p = Path(path_str)
    if not p.exists():
        return None

    arr = np.load(p)
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D XD feature array at {p}, got {arr.shape}")

    return arr.astype(np.float32)

def build_xd_fused_bag(row, num_segments: int, stream_out_dim: int) -> np.ndarray:
    """Build one fused RGB+Flow bag for a single XD sample."""
    rgb = load_stream(row.get("rgb_path"))
    flow = load_stream(row.get("flow_path"))

    if rgb is None and flow is None:
        raise FileNotFoundError(f"Both RGB and Flow missing for clip {row['clip_id']}")

    if rgb is None:
        flow_proj = project_stream(temporal_subsample(flow, num_segments), stream_out_dim)
        rgb_proj = np.zeros_like(flow_proj)
    elif flow is None:
        rgb_proj = project_stream(temporal_subsample(rgb, num_segments), stream_out_dim)
        flow_proj = np.zeros_like(rgb_proj)
    else:
        rgb_proj = project_stream(temporal_subsample(rgb, num_segments), stream_out_dim)
        flow_proj = project_stream(temporal_subsample(flow, num_segments), stream_out_dim)

    fused = np.concatenate([rgb_proj, flow_proj], axis=1).astype(np.float32)
    assert fused.shape == (num_segments, stream_out_dim * 2), fused.shape
    return fused

# ============================================================================
# 3. BAG WRITER
# ============================================================================

def build_xd_bags_for_df(df, out_dir: Path, cap_per_class=None, overwrite: bool = False) -> int:
    """Build and save fused bags for a dataframe split."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if len(df) == 0:
        return 0

    subset = df
    if cap_per_class is not None:
        parts = [df[df["label"] == lab].head(cap_per_class) for lab in sorted(df["label"].unique())]
        subset = pd.concat(parts, ignore_index=True)

    n_saved = 0

    for _, row in tqdm(subset.iterrows(), total=len(subset), desc="XD RGB+Flow bags"):
        out_path = out_dir / f"{row['clip_id']}.npy"
        out_path.parent.mkdir(parents=True, exist_ok=True)

        if out_path.exists() and not overwrite:
            n_saved += 1
            continue

        bag = build_xd_fused_bag(row, NUM_SEGMENTS, _stream_out_dim)
        np.save(out_path, bag)
        n_saved += 1

    return n_saved

# ============================================================================
# 4. BUILD TRAIN / VAL / TEST BAGS
# ============================================================================

_saved_train = build_xd_bags_for_df(xd_train_df, FEATURES_DIR_XD, XD_FEATURE_CAP_PER_CLASS)
_saved_val = build_xd_bags_for_df(xd_val_df, FEATURES_DIR_XD, XD_FEATURE_CAP_PER_CLASS)
_saved_test = build_xd_bags_for_df(xd_test_df, FEATURES_DIR_XD, XD_FEATURE_CAP_PER_CLASS)

print(f"Saved XD fused bags -> train={_saved_train}, val={_saved_val}, test={_saved_test}, out_dim={_total_xd_out_dim}")

BUILD XD RGB + FLOW FUSED BAGS
Total XD output dim: 512
Per-stream output dim: 256


XD RGB+Flow bags: 100%|██████████| 4000/4000 [00:00<00:00, 41456.55it/s]

Saved XD fused bags -> train=15816, val=3954, test=4000, out_dim=512


## 5. Sanity check

This section verifies that the dataset preparation and configuration are consistent before continuing.

It checks:

- selected feature mode
- whether 3D features are enabled
- whether fusion mode is enabled
- final MIL input dimension
- CLIP semantic extension settings
- UCF data path
- XD I3D feature path
- XD paired metadata status
- XD train/validation/test row counts

This step helps detect configuration errors early, such as missing folders, incorrect feature dimensions, stale metadata, or an inconsistent dataset mode.


In [10]:
# ============================================================================
# SANITY CHECK
# Verifies config, dataset paths, and paired XD metadata status
# ============================================================================

print("=" * 70)
print("SANITY CHECK")
print("=" * 70)

# ============================================================================
# 1. CONFIG CHECK
# ============================================================================

print("feature_mode =", CFG["feature_mode"])
print("USE_3D_FEATURES =", USE_3D_FEATURES)
print("USE_FUSION =", USE_FUSION)
print("MIL_INPUT_DIM =", MIL_INPUT_DIM)
print("USE_CLIP_TEXT_EXTRA_CREDIT =", USE_CLIP_TEXT_EXTRA_CREDIT)
print("ANOMALY_LABEL_SET =", ANOMALY_LABEL_SET)

# ============================================================================
# 2. PATH CHECK
# ============================================================================

print("DATA_ROOT exists:", Path(DATA_ROOT).exists())
print("I3D_FEATURES_DIR exists:", Path(I3D_FEATURES_DIR).exists())
print("FEATURES_DIR_XD exists:", Path(FEATURES_DIR_XD).exists())

# ============================================================================
# 3. XD PAIRED DATA CHECK
# ============================================================================

if "xd_metadata_df" in globals():
    paired_count = int(((xd_metadata_df["rgb_path"].notna()) & (xd_metadata_df["flow_path"].notna())).sum())
    print("XD paired RGB+Flow rows:", paired_count, "out of", len(xd_metadata_df))

if "xd_train_df" in globals():
    print("XD train rows:", len(xd_train_df))
if "xd_val_df" in globals():
    print("XD val rows:", len(xd_val_df))
if "xd_test_df" in globals():
    print("XD test rows:", len(xd_test_df))

SANITY CHECK
feature_mode = 3d
USE_3D_FEATURES = True
USE_FUSION = False
MIL_INPUT_DIM = 512
USE_CLIP_TEXT_EXTRA_CREDIT = True
ANOMALY_LABEL_SET = ['normal', 'abuse', 'arrest', 'arson', 'assault', 'burglary', 'explosion', 'fighting', 'road accident', 'robbery', 'shooting', 'shoplifting', 'stealing', 'vandalism', 'violence']
DATA_ROOT exists: True
I3D_FEATURES_DIR exists: True
FEATURES_DIR_XD exists: True
XD paired RGB+Flow rows: 23770 out of 23770
XD train rows: 15816
XD val rows: 3954
XD test rows: 4000


## 6. UCF-Crime frame metadata builder

This section scans the UCF-Crime PNG frame dataset and groups frames into video clips.

The code recursively finds `.png` files under `DATA_ROOT` and extracts:

```text
clip_id
split
class_name
label
num_frames
frame_paths
dataset
```

### Clip grouping

Frames are grouped by filename prefix.

Example:

```text
Arson042_x264_1170.png
Arson042_x264_1560.png
```

Both frames are assigned to:

```text
clip_id = Arson042_x264
```

The final number in the filename is treated as the frame index, so frames can be sorted in temporal order.

### Label rule

```text
NormalVideos → label 0
Any anomaly class → label 1
```

### Output summary

The submitted run found:

```text
1900 UCF clips
950 normal clips
950 anomalous clips
1610 official train clips
290 official test clips
```

This converts UCF-Crime from individual frame files into clip-level metadata suitable for weakly supervised MIL.


In [11]:
# ============================================================================
# BUILD UCF-CRIME FRAME METADATA
# Scans PNG frames under DATA_ROOT, groups them into clips,
# infers split / class / label, and caches metadata.
# Produces:
# - metadata_df
# ============================================================================

# --- Phases 2+ guarded: set PHASE_ACTIVE >= 2 in Config to run Sections 3 onward ---
if PHASE_ACTIVE < 2:
    raise RuntimeError("Phase 1 only: set PHASE_ACTIVE = 2 in Config (Section 2) to run Sections 3+.")

from pathlib import Path
import pickle
import pandas as pd
from collections import defaultdict
from tqdm import tqdm

print("=" * 70)
print("BUILD UCF-CRIME FRAME METADATA")
print("=" * 70)

# ============================================================================
# 1. CACHE PATH
# ============================================================================

_cache_path = METADATA_DIR / "ucf_frame_metadata.pkl"
use_cache = False

# ============================================================================
# 2. LOAD CACHE IF AVAILABLE
# ============================================================================

if _cache_path.exists():
    with open(_cache_path, "rb") as f:
        data = pickle.load(f)

    if isinstance(data, dict) and "data_root_mtime" in data:
        try:
            current_mtime = Path(DATA_ROOT).stat().st_mtime

            if current_mtime == data["data_root_mtime"]:
                metadata_df = pd.DataFrame(data["records"])

                if "label" not in metadata_df.columns:
                    use_cache = False
                    print("Cache missing 'label' column; rescanning.")

                elif len(metadata_df) > 0 and (metadata_df["label"] == 0).sum() == 0:
                    use_cache = False
                    print("Cache has no normal clips; rescanning to pick up NormalVideos.")

                else:
                    use_cache = True
                    print("Loaded", len(metadata_df), "clips from cache (data unchanged).")

            else:
                print("DATA_ROOT changed since cache; rescanning.")

        except OSError:
            print("Could not stat DATA_ROOT; rescanning.")

    else:
        # Legacy cache handling
        metadata_df = pd.DataFrame(data)

        if len(metadata_df) == 0 or (metadata_df.columns.empty if hasattr(metadata_df, "columns") else True):
            metadata_df = pd.DataFrame(columns=[
                "dataset", "clip_id", "split", "class_name",
                "label", "num_frames", "frame_paths"
            ])
            use_cache = True
            print("Loaded 0 clips from cache (legacy, empty).")

        elif "label" not in metadata_df.columns:
            use_cache = False
            print("Cache missing 'label' column; rescanning.")

        elif len(metadata_df) > 0 and (metadata_df["label"] == 0).sum() == 0:
            use_cache = False
            print("Cache has no normal clips; rescanning to pick up NormalVideos.")

        else:
            use_cache = True
            print("Loaded", len(metadata_df), "clips from cache (legacy format).")

# ============================================================================
# 3. BUILD METADATA FROM PNG FRAMES
# ============================================================================

if not use_cache:

    def build_ucf_frame_metadata(root_dir: str) -> pd.DataFrame:
        """
        Scan PNG frames, group them by clip, and build metadata rows.

        Expected rough structure:
        DATA_ROOT/
            Train/
                NormalVideos/
                Arson/
                ...
            Test/
                NormalVideos/
                Arson/
                ...
        """
        root = Path(root_dir)

        if not root.exists():
            raise FileNotFoundError(f"DATA_ROOT does not exist: {root}")

        clips = defaultdict(list)

        # ----------------------------------------------------------------------
        # 3.1 Scan PNG files and group by clip_id
        # ----------------------------------------------------------------------
        for p in tqdm(root.rglob("*.png"), desc="Scanning UCF PNGs"):
            stem = p.stem
            parts = stem.rsplit("_", 1)

            clip_id = parts[0] if len(parts) == 2 else stem
            frame_idx = int(parts[1]) if len(parts) == 2 and parts[1].isdigit() else 0

            clips[clip_id].append((frame_idx, p))

        # ----------------------------------------------------------------------
        # 3.2 Build dataframe rows
        # ----------------------------------------------------------------------
        rows = []

        for clip_id, frame_list in clips.items():
            frame_list.sort(key=lambda x: x[0])

            paths = [str(p) for _, p in frame_list]
            first_path = frame_list[0][1]

            parent_name = first_path.parent.name
            grandparent = first_path.parent.parent.name

            split = grandparent if grandparent in ("Train", "Test") else "Train"
            class_name = parent_name
            label = 0 if "Normal" in parent_name or "Normal" in str(first_path) else 1

            rows.append({
                "dataset": "ucf_crime",
                "clip_id": clip_id,
                "split": split,
                "class_name": class_name,
                "label": label,
                "num_frames": len(paths),
                "frame_paths": paths,
            })

        return pd.DataFrame(rows)

    metadata_df = build_ucf_frame_metadata(DATA_ROOT)

    # --------------------------------------------------------------------------
    # 3.3 Handle empty scan safely
    # --------------------------------------------------------------------------
    if len(metadata_df) == 0:
        metadata_df = pd.DataFrame(columns=[
            "dataset", "clip_id", "split", "class_name",
            "label", "num_frames", "frame_paths"
        ])
        print("No clips found. Check DATA_ROOT and that PNGs exist under Train/ / Test/ folders.")

    # --------------------------------------------------------------------------
    # 3.4 Save cache
    # --------------------------------------------------------------------------
    METADATA_DIR.mkdir(parents=True, exist_ok=True)

    if Path(DATA_ROOT).exists():
        data_root_mtime = Path(DATA_ROOT).stat().st_mtime
    else:
        data_root_mtime = None

    with open(_cache_path, "wb") as f:
        pickle.dump({
            "records": metadata_df.to_dict("records"),
            "data_root_mtime": data_root_mtime
        }, f)

    print("Clips:", len(metadata_df), "(saved to cache; will reuse until DATA_ROOT changes).")

# ============================================================================
# 4. SUMMARY CHECKS
# ============================================================================

if len(metadata_df) > 0:
    print("Split counts:", metadata_df["split"].value_counts().to_dict())
    print("Label counts:", metadata_df["label"].value_counts().to_dict())
    print("Class counts (top 10):", metadata_df["class_name"].value_counts().head(10).to_dict())
else:
    print("metadata_df is empty.")

# ============================================================================
# 5. PREVIEW
# ============================================================================

if len(metadata_df) > 0:
    display(metadata_df[["clip_id", "split", "class_name", "label", "num_frames"]].head())
else:
    display(metadata_df)

BUILD UCF-CRIME FRAME METADATA
Loaded 1900 clips from cache (data unchanged).
Split counts: {'Train': 1610, 'Test': 290}
Label counts: {1: 950, 0: 950}
Class counts (top 10): {'NormalVideos': 950, 'RoadAccidents': 150, 'Robbery': 150, 'Burglary': 100, 'Stealing': 100, 'Vandalism': 50, 'Arrest': 50, 'Arson': 50, 'Shooting': 50, 'Explosion': 50}


,clip_id,split,class_name,label,num_frames
0,Vandalism023_x264,Train,Vandalism,1,631
1,Vandalism006_x264,Train,Vandalism,1,82
2,Vandalism041_x264,Train,Vandalism,1,487
3,Vandalism031_x264,Train,Vandalism,1,180
4,Vandalism008_x264,Train,Vandalism,1,1259


## 7. UCF metadata persistence

This section saves or reloads the UCF metadata cache:

```text
ucf_frame_metadata.pkl
```

### Purpose of caching

Scanning over one million PNG frames can take time. Once the metadata has been built, the cache allows the notebook to reload the clip table quickly after a restart.

### Cached content

The cache stores clip-level metadata:

```text
clip_id
split
class_name
label
num_frames
ordered frame_paths
```

The code also performs safety checks to detect stale or invalid metadata. For example, if the cache contains no normal clips, the notebook can warn that the metadata should be rebuilt.


In [12]:
# Save/load UCF frame manifest (ucf_frame_metadata.pkl). Run after Section 3 to persist or reload metadata_df.
import pickle
from pathlib import Path
import pandas as pd

if 'METADATA_DIR' not in globals():
    try:
        METADATA_DIR = Path(PROCESSED_ROOT) / "metadata"
    except NameError:
        raise NameError("Run the Configuration cell (Section 2) first to set METADATA_DIR and PROCESSED_ROOT.")

def save_ucf_frame_manifest(df: pd.DataFrame, metadata_dir: Path) -> None:
    metadata_dir = Path(metadata_dir)
    metadata_dir.mkdir(parents=True, exist_ok=True)
    pkl_path = metadata_dir / "ucf_frame_metadata.pkl"
    with open(pkl_path, "wb") as f:
        pickle.dump(df.to_dict("records"), f)
    print("Saved:", pkl_path)

def load_ucf_frame_manifest(metadata_dir: Path) -> pd.DataFrame:
    pkl_path = Path(metadata_dir) / "ucf_frame_metadata.pkl"
    if not pkl_path.exists():
        raise FileNotFoundError(f"Run metadata builder first. Not found: {pkl_path}")
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)
    if isinstance(data, dict) and "records" in data:
        records = data["records"]
    else:
        records = data
    return pd.DataFrame(records)

MANIFEST_PATH = METADATA_DIR / "ucf_frame_metadata.pkl"
if MANIFEST_PATH.exists():
    metadata_df = load_ucf_frame_manifest(METADATA_DIR)
    print("Loaded", len(metadata_df), "clips from manifest.")
else:
    if "metadata_df" not in dir():
        raise NameError("Run Section 3 (Frame metadata builder) first to create metadata_df.")
    save_ucf_frame_manifest(metadata_df, METADATA_DIR)
    print("Saved manifest.")
if len(metadata_df) > 0 and "label" in metadata_df.columns:
    display(metadata_df["label"].value_counts())
    n_normal = (metadata_df["label"] == 0).sum()
    if n_normal == 0:
        print("ERROR: No normal clips (label 0). Sultani loss needs BOTH classes. Ensure DATA_ROOT has Train/NormalVideos/ with PNGs, then run \"Clear UCF cache\" cell and re-run this cell.")
else:
    print("No clips in metadata (run Section 3 after Cell 1c copy so DATA_ROOT has PNGs).")


Loaded 1900 clips from manifest.


label
1    950
0    950
Name: count, dtype: int64

## 8. Train / validation / test split

This section creates the final UCF-Crime train, validation and test splits.

The final UCF split is not a random 80/10/10 split over the whole dataset. Instead, the official UCF-Crime test set is kept unchanged:

```text
Official Train clips → split into train and validation
Official Test clips → kept as test
```

This respects the original dataset organisation and avoids test leakage.

### Submitted UCF split sizes

The submitted run produced:

```text
Train = 1420 clips
Validation = 190 clips
Test = 290 clips
```

Label counts:

```text
Train: {anomaly: 714, normal: 706}
Validation: {anomaly: 96, normal: 94}
Test: {normal: 150, anomaly: 140}
```

### Purpose of the validation set

The validation set is used for model selection, early stopping, hyperparameter comparison, and comparison of Baseline MIL against Attention MIL before final test evaluation.

### Clip-level split

Splitting is performed at clip/video level rather than frame level. This prevents frames from the same video appearing in both training and evaluation sets.


In [13]:
# ============================================================================
# BUILD UCF TRAIN / VAL / TEST SPLITS
# Uses official UCF split from metadata:
# - Keep official Test split unchanged
# - Split official Train split into train / val only
# Produces:
# - train_df
# - val_df
# - test_df
# ============================================================================

if PHASE_ACTIVE < 2:
    raise RuntimeError("Phase 1 only. Set PHASE_ACTIVE = 2 in Config to run pipeline.")

import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

print("=" * 70)
print("BUILD UCF TRAIN / VAL / TEST SPLITS")
print("=" * 70)

# ============================================================================
# 1. VALIDATION CHECKS
# ============================================================================

if "metadata_df" not in globals():
    raise RuntimeError("metadata_df not found. Run the UCF metadata cell first.")

if len(metadata_df) == 0:
    raise RuntimeError("metadata_df is empty. Check UCF metadata building step first.")

required_cols = {"dataset", "clip_id", "split", "class_name", "label", "num_frames", "frame_paths"}
missing_cols = required_cols - set(metadata_df.columns)
if missing_cols:
    raise KeyError(f"metadata_df missing required columns: {missing_cols}")

# ============================================================================
# 2. SEPARATE OFFICIAL TRAIN / TEST
# ============================================================================

official_train_df = metadata_df[metadata_df["split"] == "Train"].copy().reset_index(drop=True)
official_test_df = metadata_df[metadata_df["split"] == "Test"].copy().reset_index(drop=True)

print("Official Train clips:", len(official_train_df))
print("Official Test clips:", len(official_test_df))

if len(official_train_df) == 0 or len(official_test_df) == 0:
    raise RuntimeError("Expected official Train/Test splits in metadata_df, but one of them is empty.")

# ============================================================================
# 3. SPLIT OFFICIAL TRAIN INTO TRAIN / VAL
# ============================================================================

train_df, val_df = train_test_split(
    official_train_df,
    test_size=0.11801242236024845,   # ≈ 190 out of 1610
    stratify=official_train_df["label"],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = official_test_df.reset_index(drop=True)

# ============================================================================
# 4. SAVE SPLITS
# ============================================================================

SPLITS_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_pickle(SPLITS_DIR / "train_df.pkl")
val_df.to_pickle(SPLITS_DIR / "val_df.pkl")
test_df.to_pickle(SPLITS_DIR / "test_df.pkl")

train_df.drop(columns=["frame_paths"], errors="ignore").to_csv(SPLITS_DIR / "train.csv", index=False)
val_df.drop(columns=["frame_paths"], errors="ignore").to_csv(SPLITS_DIR / "val.csv", index=False)
test_df.drop(columns=["frame_paths"], errors="ignore").to_csv(SPLITS_DIR / "test.csv", index=False)

# XD splits for Notebook 2 (DATASET_MODE xd / both)
if "xd_train_df" in globals() and "xd_val_df" in globals() and "xd_test_df" in globals():
    xd_train_df.to_pickle(SPLITS_DIR / "xd_train_df.pkl")
    xd_val_df.to_pickle(SPLITS_DIR / "xd_val_df.pkl")
    xd_test_df.to_pickle(SPLITS_DIR / "xd_test_df.pkl")
    print("Saved xd_train_df.pkl / xd_val_df.pkl / xd_test_df.pkl for Notebook 2.")
elif DATASET_MODE in ("xd", "both"):
    print("Warning: xd_* splits not in memory. Run XD metadata + split cells in Task 1 first, then re-run this cell.")

# ============================================================================
# 5. SUMMARY
# ============================================================================

print("Final split sizes:")
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

print("\nLabel counts:")
print("Train:", train_df["label"].value_counts().to_dict())
print("Val:", val_df["label"].value_counts().to_dict())
print("Test:", test_df["label"].value_counts().to_dict())

if len(train_df) == 0:
    print("No train clips. Run UCF metadata cell first, then this cell.")

BUILD UCF TRAIN / VAL / TEST SPLITS
Official Train clips: 1610
Official Test clips: 290
Saved xd_train_df.pkl / xd_val_df.pkl / xd_test_df.pkl for Notebook 2.
Final split sizes:
Train: 1420 Val: 190 Test: 290

Label counts:
Train: {1: 714, 0: 706}
Val: {1: 96, 0: 94}
Test: {0: 150, 1: 140}


## Troubleshooting: zero normal clips

If the training split shows zero normal clips, the most likely cause is a stale or incorrect metadata cache.

Check that the UCF-Crime folder contains:

```text
Train/NormalVideos/
Test/NormalVideos/
```

Then delete the cached metadata file:

```text
ucf_frame_metadata.pkl
```

from `METADATA_DIR`, and rerun the UCF metadata builder.

This check is important because the MIL ranking loss requires both normal and anomalous videos. The model learns by comparing abnormal bags against normal bags, so both classes must be present in the training data.


## 9. Notebook 1 → Notebook 2 handoff

After metadata and splits are created, this section writes:

```text
artifacts/task1_session.json
```

This file stores the important paths and configuration values needed by Notebook 2.

### Saved information

The session file includes:

```text
project_root
processed_root
metadata_dir
splits_dir
feature directories
dataset_mode
feature_mode
num_segments
seed
CLIP settings
anomaly label set
input feature dimensions
```

### Purpose of the handoff file

Notebook 2 starts from a fresh kernel. The handoff file ensures that Notebook 2 reloads the same paths, dataset settings, feature settings and CLIP configuration produced in Notebook 1.

This keeps the multi-notebook pipeline consistent and reproducible.


In [14]:
# Persist Task 1 paths + config for Notebook 2 (run after splits are saved)
import json
import os
from pathlib import Path

def _submission_root() -> Path:
    env = os.environ.get("VAD_SUBMISSION_ROOT")
    if env:
        return Path(env).resolve()
    cwd = Path.cwd().resolve()
    if cwd.name == "notebooks":
        return cwd.parent
    return cwd

SUBMISSION_ROOT = _submission_root()
ARTIFACTS_DIR = SUBMISSION_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

session = {
    "version": 1,
    "project_root": str(Path(PROJECT_ROOT).resolve()),
    "processed_root": str(Path(PROCESSED_ROOT).resolve()),
    "data_root": str(Path(DATA_ROOT).resolve()),
    "xd_violence_root": str(Path(XD_VIOLENCE_ROOT).resolve()),
    "i3d_features_dir": str(Path(I3D_FEATURES_DIR).resolve()),
    "metadata_dir": str(METADATA_DIR.resolve()),
    "splits_dir": str(SPLITS_DIR.resolve()),
    "features_dir": str(FEATURES_DIR.resolve()),
    "features_dir_xd": str(FEATURES_DIR_XD.resolve()),
    "features_3d_dir": str(FEATURES_3D_DIR.resolve()),
    "features_fusion_dir": str(FEATURES_FUSION_DIR.resolve()),
    "results_dir": str(RESULTS_DIR.resolve()),
    "use_runs_dir": USE_RUNS_DIR,
    "run_tag": RUN_TAG,
    "run_id": Path(PROJECT_ROOT).name if USE_RUNS_DIR else None,
    "cfg": CFG,
    "seed": SEED,
    "num_segments": NUM_SEGMENTS,
    "dataset_mode": DATASET_MODE,
    "use_fusion": USE_FUSION,
    "use_3d_features": USE_3D_FEATURES,
    "use_clip_text_extra_credit": USE_CLIP_TEXT_EXTRA_CREDIT,
    "anomaly_label_set": ANOMALY_LABEL_SET,
    "feature_dim": FEATURE_DIM,
    "feature_dim_3d": FEATURE_DIM_3D,
    "xd_feature_dim": XD_FEATURE_DIM,
    "xd_segments_raw": XD_SEGMENTS_RAW,
    "xd_stream_out_dim": XD_STREAM_OUT_DIM,
    "frames_per_segment": FRAMES_PER_SEGMENT,
    "resize_hw": list(RESIZE_HW),
    "clip_len": CLIP_LEN,
    "clip_stride": CLIP_STRIDE,
    "train_ratio": TRAIN_RATIO,
    "val_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
    "in_colab": IN_COLAB,
    "phase_active": PHASE_ACTIVE,
    "use_integrated_clip_semantic_head": USE_INTEGRATED_CLIP_SEMANTIC_HEAD,
    "clip_semantic_loss_weight": CLIP_SEMANTIC_LOSS_WEIGHT,
    "clip_model_name": CLIP_MODEL_NAME,
    "clip_pretrained": CLIP_PRETRAINED,
    "clip_text_encoder_trainable": CLIP_TEXT_ENCODER_TRAINABLE,
}

out_path = ARTIFACTS_DIR / "task1_session.json"
out_path.write_text(json.dumps(session, indent=2, sort_keys=True), encoding="utf-8")
print("Wrote:", out_path)
print("Submission root:", SUBMISSION_ROOT)

print("CLIP extra credit enabled:", USE_CLIP_TEXT_EXTRA_CREDIT)
print("Anomaly text labels:", len(ANOMALY_LABEL_SET))

print("Integrated CLIP semantic head:", USE_INTEGRATED_CLIP_SEMANTIC_HEAD)
print("CLIP semantic loss weight:", CLIP_SEMANTIC_LOSS_WEIGHT)
print("CLIP model:", CLIP_MODEL_NAME, "| pretrained:", CLIP_PRETRAINED)
print("CLIP text encoder trainable:", CLIP_TEXT_ENCODER_TRAINABLE)


Wrote: /scratch/VAD/artifacts/task1_session.json
Submission root: /scratch/VAD
CLIP extra credit enabled: True
Anomaly text labels: 15
Integrated CLIP semantic head: True
CLIP semantic loss weight: 0.1
CLIP model: ViT-B-32 | pretrained: openai
CLIP text encoder trainable: False
